# SciDCC Preprocessing

Input: `data/processed/scidcc_clean.csv` from EDA notebook.

Output: tokenized HuggingFace `DatasetDict` (train/val/test) ready for `Trainer` + class weights imbalance handling.

## 1. Setup & loading

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

SEED = 42
PROCESSED_DIR = Path('..') / 'data' / 'processed'

TOKENIZER_MODEL = 'P0L3/SciClimateBERT'
MAX_LENGTH = 512

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'scidcc_clean.csv')

with open(PROCESSED_DIR / 'label_map.json', 'r', encoding='utf-8') as f:
    label_map = json.load(f)
label2id = label_map['label2id']
id2label = {int(k): v for k, v in label_map['id2label'].items()}

print(f'Shape: {df.shape}')
print(f'Classes:  {len(label2id)}')
df['label'] = df['Category'].map(label2id)
assert df['label'].notna().all(), 'missing labels'
df['label'] = df['label'].astype(int)
df[['Category', 'label']].head(3)

## 2. Stratified split 70/15/15

In [ ]:
# train vs temp split (85-15)
train_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED
)
# 15/85 val from train
train_df, val_df = train_test_split(
    train_df, test_size=15/85, stratify=train_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df):>5}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val:   {len(val_df):>5}  ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test:  {len(test_df):>5}  ({len(test_df)/len(df)*100:.1f}%)')

In [ ]:
# all classes in every split
split_counts = pd.DataFrame({
    'train': train_df['Category'].value_counts(),
    'val':   val_df['Category'].value_counts(),
    'test':  test_df['Category'].value_counts(),
}).fillna(0).astype(int)
split_counts['total'] = split_counts.sum(axis=1)
split_counts = split_counts.sort_values('total', ascending=False)
split_counts

In [ ]:
# min samples in smalles class
print(f'Min in val:  {split_counts["val"].min()}  (classes: {split_counts["val"].idxmin()})')
print(f'Min in test: {split_counts["test"].min()}  (classes: {split_counts["test"].idxmin()})')

## 3. Preparing input variants

In [ ]:
def build_title_summary(row):
    title = str(row['Title']).strip()
    summary = str(row['Summary']).strip()
    return f'{title}. {summary}' if summary else title

def build_body(row):
    return str(row['Body']).strip()

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_df['text_title_summary'] = split_df.apply(build_title_summary, axis=1)
    split_df['text_body']          = split_df.apply(build_body, axis=1)

train_df[['Title', 'Summary', 'text_title_summary']].head(2)

## 4. Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
print(f'Tokenizer: {TOKENIZER_MODEL}')
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Max model input: {tokenizer.model_max_length}')

In [ ]:
def make_hf_dataset(df, text_col):
    out = df[[text_col, 'label']].rename(columns={text_col: 'text'})
    return Dataset.from_pandas(out, preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

def build_dataset_dict(text_col):
    raw = DatasetDict({
        'train': make_hf_dataset(train_df, text_col),
        'val':   make_hf_dataset(val_df,   text_col),
        'test':  make_hf_dataset(test_df,  text_col),
    })
    tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['text'])
    return tokenized

In [ ]:
ds_title_summary = build_dataset_dict('text_title_summary')
ds_body          = build_dataset_dict('text_body')

print('title_summary:')
print(ds_title_summary)
print('\nbody:')
print(ds_body)

In [ ]:
for name, ds in [('title_summary', ds_title_summary), ('body', ds_body)]:
    lens = [len(x) for x in ds['train']['input_ids']]
    print(f'{name:15s}  mean={np.mean(lens):.1f}  p95={np.percentile(lens, 95):.0f}  max={np.max(lens)}')

## 5. Class weights

In [ ]:
classes = np.arange(len(label2id))
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label'].values,
)

weights_df = pd.DataFrame({
    'class_id': classes,
    'category': [id2label[i] for i in classes],
    'train_count': [int((train_df['label'] == i).sum()) for i in classes],
    'weight': weights.round(4),
}).sort_values('train_count')
weights_df

## 6. Saving

In [ ]:
splits_dir = PROCESSED_DIR / 'splits'
splits_dir.mkdir(exist_ok=True, parents=True)

keep_cols = ['Date', 'Link', 'Title', 'Summary', 'Body', 'Category', 'Year', 'label']
train_df[keep_cols].to_csv(splits_dir / 'train.csv', index=False)
val_df[keep_cols].to_csv(splits_dir / 'val.csv', index=False)
test_df[keep_cols].to_csv(splits_dir / 'test.csv', index=False)
print(f'CSV splits saved to {splits_dir}')

ds_title_summary.save_to_disk(str(PROCESSED_DIR / 'hf_title_summary'))
ds_body.save_to_disk(str(PROCESSED_DIR / 'hf_body'))
print(f'HF datasets saved')

with open(PROCESSED_DIR / 'class_weights.json', 'w', encoding='utf-8') as f:
    json.dump({
        'classes':  [int(c) for c in classes],
        'weights':  [float(w) for w in weights],
        'scheme':   'balanced',
        'computed_on': 'train_split',
        'seed': SEED,
    }, f, indent=2)
print('Class weights saved')

In [ ]:
from datasets import load_from_disk
check = load_from_disk(str(PROCESSED_DIR / 'hf_title_summary'))
print(check)
print('\Example:')
ex = check['train'][0]
print(f'  label:       {ex["label"]} ({id2label[ex["label"]]})')
print(f'  input_ids:   {ex["input_ids"][:20]}...')
print(f'  length:      {len(ex["input_ids"])}')
print(f'  decoded:     {tokenizer.decode(ex["input_ids"])[:200]}...')